# CSGM SR ×4 test benchmark: correct vs wrong label vs bicubic

Analysis of the overnight run (`scripts/run_sr4_csgm_test.py`, version v2).

**Setup**

| | |
|---|---|
| Test images | 100 held-out **test** images, from 100 different classes: the fixed sample `sr4_test_sample_100_seed2028.npz` |
| Method (frozen) | CSGM-like reconstruction with the pretrained drift generator `hf://pixel_B_sota`; Adam lr 0.02, 5,000 steps, 2 restarts, no regularizer |
| Conditions per image | **correct label** (oracle class) and **wrong label** (one random other class, fixed per image) |
| Starting point | the same `eps` / `noise_labels` start for both conditions, so only the label differs |
| Restart rule | keep the restart with the lowest final LR loss (doesn't use the HR image) |
| Baseline | bicubic upsampling of the same 64×64 measurement |

**Rules.** Nothing here is tuned. The method and sample were fixed before the run, and these results must not be used to change CSGM or to tune IADrift.

This notebook only reads saved results and runs on the CPU. With a partial run, it analyzes the finished images and labels the results **PRELIMINARY**.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cpu"  # analysis only: never touch the GPU
SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import csv
import json
import math

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from scipy import stats

from analyze_sr4_csgm_test import find_results, psnr_of_mean_mse
from freq_metrics import BANDS, band_snr_db
from plot_sr4_csgm_test import DATA_DIR, load_hr256, to_display

%matplotlib inline
# (plot_sr4_csgm_test selects the non-interactive Agg backend; switch back for inline figures)

BAND_NAMES = [name for name, *_ in BANDS]
CLASS_INDEX = json.loads((DATA_DIR / "imagenet_class_index.json").read_text())


def class_name(label):
    return CLASS_INDEX[str(int(label))][1].replace("_", " ")


print("JAX backend:", jax.default_backend())

## Load and validate the results

Checks that every saved run belongs to the manifest's plan, with no duplicates and the frozen settings, and that every image is in the **test** split. Then, for each finished image and condition, it keeps the restart with the lowest LR loss.

In [ ]:
NUM_IMAGES = 100

results_path = find_results(NUM_IMAGES)
arrays_dir = Path(str(results_path.with_suffix("")) + "_arrays")
plan = json.loads(results_path.with_suffix(".manifest.json").read_text())["run_plan"]
rows = [json.loads(line) for line in results_path.open() if line.strip()]


def key(r):
    return (r["image_index"], r["condition"], r["restart"])


plan_by_key = {key(r): r for r in plan}
assert len({key(r) for r in rows}) == len(rows), "duplicate runs"
for r in rows:
    p = plan_by_key[key(r)]  # KeyError = a run that is not in the plan
    for field in ("learning_rate", "num_steps", "num_restarts", "class_label", "true_label"):
        assert r[field] == p[field], (key(r), field)
    assert math.isfinite(r["final_loss"]) and math.isfinite(r["psnr"])

split = np.load(DATA_DIR / "imageNetV2_split_80_20_seed2026.npz")
planned_images = list(dict.fromkeys(r["image_index"] for r in plan))
assert np.all(np.isin(planned_images, split["test_indices"])), "non-test image in plan"
assert not np.any(np.isin(planned_images, split["train_indices"]))

runs_per_image = len(plan) // len(planned_images)
done = {}
for r in rows:
    done.setdefault(r["image_index"], []).append(r)
images = [i for i in planned_images if len(done.get(i, [])) == runs_per_image]
complete = len(images) == len(planned_images)

best = {}
for i in images:
    for cond in ("oracle", "wrong"):
        best[(i, cond)] = min((r for r in done[i] if r["condition"] == cond),
                              key=lambda r: (r["final_loss"], r["restart"]))

print(f"{results_path.name}")
print(f"Runs saved: {len(rows)}/{len(plan)}; images complete: {len(images)}/{len(planned_images)} (all in the test split)")
if not complete:
    display(Markdown(f"**PRELIMINARY: only {len(images)} of {len(planned_images)} images are finished.**"))

## Main results

- **PSNR** is in dB, against the ground-truth 256×256 HR image (higher is better).
- **Mean per-image PSNR** averages each image's PSNR. **PSNR(mean MSE)** first averages MSE over images, then converts to dB; this is the IAGAN paper's convention.
- **Clipped** limits reconstructions to [-1, 1] first. The main metric is unclipped.
- **LR loss** is $\|y - A\hat x\|^2$ against the 64×64 measurement, i.e. how well the reconstruction explains what was observed.

The paired comparisons use the same images on both sides. Each gives the mean difference with a t-based 95% CI, a Wilcoxon signed-rank test, and how often the first method wins.

In [ ]:
def col(cond, field):
    return np.array([best[(i, cond)][field] for i in images])


psnr = {"bicubic": col("oracle", "bicubic_psnr"), "CSGM, correct label": col("oracle", "psnr"),
        "CSGM, wrong label": col("wrong", "psnr")}
psnr_clipped = {"CSGM, correct label": col("oracle", "psnr_clipped"), "CSGM, wrong label": col("wrong", "psnr_clipped")}
lr_loss = {"CSGM, correct label": col("oracle", "final_loss"), "CSGM, wrong label": col("wrong", "final_loss")}

lines = ["| Method | Mean PSNR | Median PSNR | SD | PSNR (mean MSE) | Mean PSNR, clipped | Mean LR loss | Median LR loss |",
         "|---|---:|---:|---:|---:|---:|---:|---:|"]
for name, v in psnr.items():
    clip = f"{psnr_clipped[name].mean():.2f}" if name in psnr_clipped else ""
    loss = (f"{lr_loss[name].mean():.2f} | {np.median(lr_loss[name]):.2f}" if name in lr_loss else " | ")
    lines.append(f"| {name} | {v.mean():.2f} | {np.median(v):.2f} | {v.std(ddof=1):.2f} | {psnr_of_mean_mse(v):.2f} | {clip} | {loss} |")
display(Markdown(f"**{len(images)} test images{'' if complete else ' (PRELIMINARY)'}**\n\n" + "\n".join(lines)))


def paired_row(name, a, b, unit="dB", higher_is_better=True):
    d = a - b
    half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d)) if len(d) > 1 else float("nan")
    p = stats.wilcoxon(d).pvalue if len(d) > 1 and np.any(d != 0) else float("nan")
    wins = int(np.sum(d > 0 if higher_is_better else d < 0))
    return f"| {name} | {d.mean():+.3f} {unit} | [{d.mean() - half:+.3f}, {d.mean() + half:+.3f}] | {np.median(d):+.3f} | {p:.2g} | {wins}/{len(d)} |"


lines = ["| Comparison (first − second) | Mean difference | 95% CI | Median | Wilcoxon p | First better |",
         "|---|---:|---:|---:|---:|---:|",
         paired_row("PSNR: correct label − wrong label", psnr["CSGM, correct label"], psnr["CSGM, wrong label"]),
         paired_row("PSNR: correct label − bicubic", psnr["CSGM, correct label"], psnr["bicubic"]),
         paired_row("PSNR: wrong label − bicubic", psnr["CSGM, wrong label"], psnr["bicubic"]),
         paired_row("LR loss: correct label − wrong label", lr_loss["CSGM, correct label"], lr_loss["CSGM, wrong label"],
                    unit="", higher_is_better=False)]
display(Markdown("**Paired comparisons** (for LR loss, lower is better, so \"first better\" counts a lower loss)\n\n" + "\n".join(lines)))

## Distributions over the test images

1. PSNR histograms per method.
2. Correct vs wrong label per image; points above the diagonal mean the correct label was better.
3. The per-image PSNR difference, correct − wrong.
4. LR loss, correct vs wrong label.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
bins = np.linspace(min(v.min() for v in psnr.values()) - 0.5, max(v.max() for v in psnr.values()) + 0.5, 25)
for (name, v), color in zip(psnr.items(), ("gray", "C0", "C1")):
    axes[0].hist(v, bins=bins, alpha=0.5, color=color, label=f"{name} (mean {v.mean():.2f})")
axes[0].set(xlabel="HR PSNR (dB)", ylabel="images", title="PSNR per method")
axes[0].legend(fontsize=8)

a, b = psnr["CSGM, correct label"], psnr["CSGM, wrong label"]
lo, hi = min(a.min(), b.min()) - 0.5, max(a.max(), b.max()) + 0.5
axes[1].scatter(b, a, s=14)
axes[1].plot([lo, hi], [lo, hi], "k--", linewidth=1)
axes[1].set(xlim=(lo, hi), ylim=(lo, hi), xlabel="wrong label PSNR (dB)", ylabel="correct label PSNR (dB)",
            title="Correct vs wrong label (above line: correct better)")

d = a - b
axes[2].hist(d, bins=25, color="C0")
axes[2].axvline(0, color="k", linewidth=1)
axes[2].axvline(d.mean(), color="C3", linestyle="--", label=f"mean {d.mean():+.2f} dB")
axes[2].set(xlabel="PSNR correct − wrong (dB)", ylabel="images", title="Per-image label effect")
axes[2].legend(fontsize=8)

la, lb = lr_loss["CSGM, correct label"], lr_loss["CSGM, wrong label"]
lo, hi = min(la.min(), lb.min()) * 0.8, max(la.max(), lb.max()) * 1.2
axes[3].scatter(lb, la, s=14, color="C2")
axes[3].plot([lo, hi], [lo, hi], "k--", linewidth=1)
axes[3].set(xscale="log", yscale="log", xlim=(lo, hi), ylim=(lo, hi), xlabel="wrong label LR loss",
            ylabel="correct label LR loss", title="LR loss (below line: correct fits better)")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Quality by frequency band

**Band SNR** (dB) = 10·log₁₀(HR content / error) per frequency band (`freq_metrics.py`).
- Higher is better. 0 dB = band left empty, negative = added detail worse than none.
- **Low** (0–0.0625) and **mid** (0.0625–0.125 cycles/pixel) are inside the 64×64 measurement. **High** and **very high** must be invented by the prior.

This cell is computed from the saved reconstructions; it takes about a minute.

In [ ]:
hr_cache, bicubic_cache, xhat_cache = {}, {}, {}
index_rows = {int(r["index"]): r for r in csv.DictReader(open(DATA_DIR / "imageNet_x4_index.csv"))}
lr_all = np.load(DATA_DIR / "imageNet_x4.npy", mmap_mode="r")
upsample = jax.jit(lambda y: jax.image.resize(y, (256, 256, 3), method="cubic", antialias=False))

band = {name: {b: [] for b in BAND_NAMES} for name in psnr}
for i in images:
    hr_cache[i] = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    bicubic_cache[i] = np.asarray(upsample(jnp.asarray(lr_all[i], jnp.float32)))
    for cond in ("oracle", "wrong"):
        with np.load(arrays_dir / best[(i, cond)]["array_file"]) as data:
            xhat_cache[(i, cond)] = data["x_hat"]
    for name, x in (("bicubic", bicubic_cache[i]), ("CSGM, correct label", xhat_cache[(i, "oracle")]),
                    ("CSGM, wrong label", xhat_cache[(i, "wrong")])):
        for b, v in band_snr_db(x, hr_cache[i]).items():
            band[name][b].append(v)

lines = ["| Method | " + " | ".join(f"SNR {b} (dB)" for b in BAND_NAMES) + " |", "|---|" + "---:|" * len(BAND_NAMES)]
for name in psnr:
    lines.append(f"| {name} | " + " | ".join(f"{np.mean(band[name][b]):.2f}" for b in BAND_NAMES) + " |")
display(Markdown("**Mean band SNR over the test images**\n\n" + "\n".join(lines)))

lines = ["| Band | Correct − wrong (mean, dB) | 95% CI | Correct better |", "|---|---:|---:|---:|"]
for b in BAND_NAMES:
    d = np.array(band["CSGM, correct label"][b]) - np.array(band["CSGM, wrong label"][b])
    half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
    lines.append(f"| {b} | {d.mean():+.2f} | [{d.mean() - half:+.2f}, {d.mean() + half:+.2f}] | {int(np.sum(d > 0))}/{len(d)} |")
display(Markdown("**Label effect per band**\n\n" + "\n".join(lines)))

## Example image

Ground-truth HR, CSGM with the correct label, CSGM with the wrong label, and bicubic upsampling. `EXAMPLE_IMAGE` defaults to the first image of the fixed test sample. Set it to any index from the per-image table below to see another.

In [ ]:
EXAMPLE_IMAGE = images[0]

i = EXAMPLE_IMAGE
o, w = best[(i, "oracle")], best[(i, "wrong")]
panels = [
    (hr_cache[i], f"Ground-truth HR\n{class_name(o['true_label'])}"),
    (xhat_cache[(i, "oracle")], f"CSGM, correct label: {class_name(o['class_label'])}\n"
                                f"PSNR {o['psnr']:.2f} dB, LR loss {o['final_loss']:.1f}"),
    (xhat_cache[(i, "wrong")], f"CSGM, wrong label: {class_name(w['class_label'])}\n"
                               f"PSNR {w['psnr']:.2f} dB, LR loss {w['final_loss']:.1f}"),
    (bicubic_cache[i], f"Bicubic upsample\nPSNR {o['bicubic_psnr']:.2f} dB"),
]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.9))
for ax, (x, title) in zip(axes, panels):
    ax.imshow(to_display(x))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.suptitle(f"Test image {i}  |  SR ×4 CSGM, lr 0.02, 5,000 steps, best of 2 restarts by LR loss "
             f"(correct r{o['restart']}, wrong r{w['restart']})", fontsize=11)
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.84, wspace=0.04)
plt.show()

## Per-image results

Sorted by the label effect, PSNR correct − wrong (dB). Use any index here as `EXAMPLE_IMAGE` above.

In [ ]:
order = np.argsort(psnr["CSGM, correct label"] - psnr["CSGM, wrong label"])[::-1]
lines = ["| # | Image | True class | Wrong label | Bicubic | Correct | Wrong | Correct − wrong | LR loss correct | LR loss wrong |",
         "|---:|---:|---|---|---:|---:|---:|---:|---:|---:|"]
for rank, k in enumerate(order, start=1):
    i = images[k]
    o, w = best[(i, "oracle")], best[(i, "wrong")]
    lines.append(f"| {rank} | {i} | {class_name(o['true_label'])} | {class_name(w['class_label'])} | {o['bicubic_psnr']:.2f} | "
                 f"{o['psnr']:.2f} | {w['psnr']:.2f} | {o['psnr'] - w['psnr']:+.2f} | {o['final_loss']:.1f} | {w['final_loss']:.1f} |")
display(Markdown("\n".join(lines)))

## Notes (100 test images, frozen method; written after the run)

**1. CSGM is far below bicubic, on every image.**
- Correct label: −3.31 dB (95% CI −3.63 to −2.99); wrong label: −3.33 dB. Neither wins on any of the 100 images.
- This is consistent with the development results (−3.9 dB on 4 images; −3.7 dB in the 100-image development sweep) and with the expectation that a CSGM-like method trails bicubic on PSNR.

**2. The class label makes no detectable difference to PSNR.**
- Correct − wrong: **+0.02 dB** (95% CI −0.24 to +0.28; Wilcoxon p = 0.71). The correct label is better on 46 of 100 images, and the median difference is −0.05 dB.
- With 100 images we can rule out an average effect larger than about ±0.3 dB.
- The development hints that a wrong label scores *higher* (1704, 8117, 403) don't hold up on the test set.

**3. Where the label does matter, it matters a lot, and mostly in one direction.**
- The per-image differences are skewed: a few images gain 2–5 dB from the correct label (wombat +5.4, barn spider +4.8, Bernese mountain dog +3.7, printer +2.3). Losses stay within about −3 dB.
- The large gains come with clearly worse LR fits under the wrong label (e.g. Bernese mountain dog: LR loss 17.6 vs 69.0 with "web site"). There, the wrong class apparently can't explain the measurement well.

**4. Measurement fit: the correct label helps slightly, at most.**
- LR loss: median −3.3 lower with the correct label, lower on 59/100 images, but not significant (p = 0.11; mean CI −10.3 to +4.8).
- Low-band SNR is +0.66 dB better with the correct label (CI +0.17 to +1.16). This is one of four band comparisons with no correction for multiple testing, so read it as a hint.
- The low-band gain doesn't show up in PSNR. Band SNR is computed with a centre-weighted (Hann) window and PSNR isn't, which may partly explain the mismatch; that's not verified.

**5. Frequency picture (same as on development images).**
- CSGM's main deficit is in the measured bands (low SNR about 22 dB vs 38.7 for bicubic; mid about 5.5 vs 8.8 dB).
- Above the LR limit CSGM is negative (high about −1.5 dB, very high about −4.1 dB), while bicubic is at about 0 dB.

**Implications.**
- For CSGM in this setting, the class prior buys essentially nothing on average in PSNR. So the predicted-class version should land close to these numbers whatever the classifier's accuracy. That's itself a reportable result.
- These numbers are the frozen **CSGM test benchmark** (correct label: 21.70 dB mean per-image PSNR, 20.66 dB from mean MSE). IADrift must be frozen on development images before it's evaluated on this same sample.

---
# Part 2 (added 2026-10-05): back-projection, LPIPS and blockiness on the test outputs

**Purpose.** Complete the test-set evaluation of the frozen CSGM benchmark (Part 1) with:
- **M2:** two post-processed methods, **bicubic + BP** and **CSGM + BP** (correct and wrong label);
- **M3:** two additional metrics for every method, **LPIPS** and **blockiness**.

**Test-set rules (why this is legitimate).**
- No new optimization is run and nothing is tuned: the CSGM outputs are the frozen benchmark's (restart with the lowest LR loss, as in Part 1).
- **Back-projection has no tunable parameters:** $x_{BP} = x + A^T (A A^T)^{-1} (y - A x)$, the image closest to $x$ that reproduces the measurement $y$ exactly, solved with conjugate gradients (tolerance 1e-6, at most 500 iterations), the same operator used on development images (notebook 04, Check 1). It uses only $y$, never HR.
- **Both metrics were defined before any test result was seen** (LPIPS in notebook 04 on development images; blockiness on 2026-10-05, sanity-checked on development outputs only).
- These results **must not influence the choice of the IADrift variant** for the test run; that choice is made on development images and written into `docs/PROTOCOL.md` first.

**Methods (all 100 test images):** bicubic, bicubic + BP, CSGM (correct label), CSGM (correct label) + BP, CSGM (wrong label), CSGM (wrong label) + BP.

**Metrics.**
- **HR PSNR** (dB, data range 2): mean of per-image PSNR, and PSNR of the mean MSE (IAGAN convention). Clipped PSNR too.
- **LPIPS** (AlexNet, `lpips` 0.1.4, CPU torch; images clipped to [-1, 1]): perceptual distance to HR, **lower is better**.
- **Blockiness** (`freq_metrics.blockiness`): mean absolute pixel jump across the generator's 16-pixel tile borders, divided by the mean jump at the other 4-pixel-aligned positions (both axes). About 1 = no tiling; > 1 = visible tile seams. Comparing only against 4-pixel-aligned positions cancels bicubic ×4's own period-4 structure. Limitation: an image that is noisy everywhere hides its seams (lower ratio), so read it together with LPIPS and the images.
- **LR loss** ‖y − A x‖² and **band SNR** as in Part 1.

**Paired comparisons** (same 100 images): mean difference with a t-based 95% CI, Wilcoxon signed-rank p, and how often the first method is better. No multiple-comparison correction.

**Visual examples:** the first 4 images of the fixed test sample (not chosen by result), full image and an 80×80 detail crop.

**Cache:** back-projected images and all metrics in `~/work/datasets/ImageNetV2/csgm_test_postproc_v1/`. The whole part runs on the CPU (about 10–15 minutes the first time).

In [ ]:
# Part 2 setup: operator, back-projection, LPIPS, blockiness (CPU only, as the rest of this notebook).
import io

import lpips
import torch
from jax.scipy.sparse.linalg import cg

from freq_metrics import blockiness

POST_DIR = DATA_DIR / "csgm_test_postproc_v1"
POST_DIR.mkdir(exist_ok=True)


def A_sr4(x):
    # Identical to run_sr4_csgm_test.A_sr4 (antialiased bicubic 256 -> 64); checked against the stored LR below.
    return jax.image.resize(x.astype(jnp.float32), (x.shape[0], 64, 64, 3), method="cubic", antialias=True)


_A_transpose = jax.linear_transpose(A_sr4, jnp.zeros((1, 256, 256, 3), jnp.float32))


@jax.jit
def _backproject(x, y):
    def AT(v):
        return _A_transpose(v)[0]
    u, _ = cg(lambda v: A_sr4(AT(v)), y - A_sr4(x), tol=1e-6, maxiter=500)
    return x + AT(u)


def backproject(x, y):
    return np.asarray(_backproject(jnp.asarray(x, jnp.float32)[None], jnp.asarray(y, jnp.float32)[None])[0])


def lr_loss(x, y):
    return float(jnp.sum((A_sr4(jnp.asarray(x, jnp.float32)[None]) - jnp.asarray(y, jnp.float32)[None]) ** 2))


def psnr_db(hr, x):
    mse = np.mean(((np.asarray(hr, np.float64) - np.asarray(x, np.float64)) / 2.0) ** 2)
    return float(-10 * np.log10(mse))


torch.set_num_threads(4)
LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).eval()


def lpips_dist(x, ref):
    to_t = lambda a: torch.from_numpy(np.ascontiguousarray(np.clip(a, -1, 1).transpose(2, 0, 1)[None])).float()
    with torch.no_grad():
        return float(LPIPS_NET(to_t(x), to_t(ref)))


def evaluate(x, hr, y):
    return {"psnr": psnr_db(hr, x), "psnr_clipped": psnr_db(hr, np.clip(x, -1, 1)), "lpips": lpips_dist(x, hr),
            "blockiness": blockiness(x), "lr_loss": lr_loss(x, y), "band_snr_db": band_snr_db(x, hr)}


# Operator check: the stored LR must equal A_sr4(HR) (Part 1 data checks found max 3.6e-7).
i = images[0]
print("max |A_sr4(HR) - stored LR|:", float(np.abs(np.asarray(A_sr4(jnp.asarray(hr_cache[i])[None])[0]) - lr_all[i]).max()))

In [ ]:
# M2 + M3: back-project and score every method on every test image (cached per image).
METHODS = ["bicubic", "bicubic + BP", "CSGM, correct label", "CSGM, correct label + BP",
           "CSGM, wrong label", "CSGM, wrong label + BP"]
post = {}
for n, i in enumerate(images):
    path = POST_DIR / f"img{i:05d}.npz"
    if path.exists():
        with np.load(path) as data:
            post[i] = json.loads(str(data["metrics_json"]))
        continue
    y, hr, bic = lr_all[i], hr_cache[i], bicubic_cache[i]
    outputs = {"bicubic": bic, "bicubic + BP": backproject(bic, y),
               "CSGM, correct label": xhat_cache[(i, "oracle")], "CSGM, wrong label": xhat_cache[(i, "wrong")]}
    outputs["CSGM, correct label + BP"] = backproject(outputs["CSGM, correct label"], y)
    outputs["CSGM, wrong label + BP"] = backproject(outputs["CSGM, wrong label"], y)
    post[i] = {"hr_blockiness": blockiness(hr), **{m: evaluate(outputs[m], hr, y) for m in METHODS}}
    buffer = io.BytesIO()
    np.savez_compressed(buffer, metrics_json=json.dumps(post[i]),
                        **{m.replace(", ", "_").replace(" + ", "_").replace(" ", "_"): outputs[m]
                           for m in METHODS if m.endswith("+ BP")})
    path.write_bytes(buffer.getvalue())  # one file per image; a crash leaves earlier images cached
    if (n + 1) % 10 == 0:
        print(f"{n + 1}/{len(images)} images done", flush=True)

# Consistency with Part 1 (same images, same PSNR definition) and the BP check.
d = max(abs(post[i]["CSGM, correct label"]["psnr"] - best[(i, "oracle")]["psnr"]) for i in images)
print(f"max |PSNR here - Part 1 PSNR| (CSGM, correct label): {d:.2e} dB")
print("max LR loss after BP:", max(post[i][m]["lr_loss"] for i in images for m in METHODS if m.endswith("+ BP")))
atomic = POST_DIR / "metrics.json"
atomic.write_text(json.dumps({str(i): post[i] for i in images}, indent=1))

In [ ]:
def values(method, key, band=None):
    return np.array([post[i][method]["band_snr_db"][band] if band else post[i][method][key] for i in images])


lines = ["| Method | Mean PSNR | PSNR (mean MSE) | Clipped | LPIPS ↓ | Blockiness (≈ 1 = none) | LR loss | "
         + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |", "|---|---:|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
hr_block = np.array([post[i]["hr_blockiness"] for i in images])
for m in METHODS:
    p = values(m, "psnr")
    lines.append(f"| {m} | {p.mean():.2f} | {psnr_of_mean_mse(p):.2f} | {values(m, 'psnr_clipped').mean():.2f} | "
                 f"{values(m, 'lpips').mean():.3f} | {values(m, 'blockiness').mean():.3f} | {values(m, 'lr_loss').mean():.2f} | "
                 + " | ".join(f"{values(m, None, b).mean():.2f}" for b in BAND_NAMES) + " |")
lines.append(f"| HR (reference) | — | — | — | 0 | {hr_block.mean():.3f} | 0 | " + " | ".join("—" for _ in BAND_NAMES) + " |")
display(Markdown(f"**All methods, mean over the {len(images)} test images**\n\n" + "\n".join(lines)))


def paired_row(a, b, key):
    lower_better = key in ("lpips", "blockiness")
    d = values(a, key) - values(b, key)
    half = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    better = int(np.sum(d < 0 if lower_better else d > 0))
    f = "{:+.3f}" if lower_better else "{:+.2f}"
    return (f"| {a} − {b} | {f.format(d.mean())} | [{f.format(d.mean() - half)}, {f.format(d.mean() + half)}] | "
            f"{p:.1e} | {better}/{len(d)} |")


PAIRS = [("bicubic + BP", "bicubic"),
         ("CSGM, correct label + BP", "CSGM, correct label"),
         ("CSGM, correct label + BP", "bicubic + BP"),
         ("CSGM, correct label", "bicubic"),
         ("CSGM, correct label + BP", "CSGM, wrong label + BP"),
         ("CSGM, wrong label + BP", "bicubic + BP")]
for key, title in (("psnr", "HR PSNR (dB; higher is better)"), ("lpips", "LPIPS (lower is better)"),
                   ("blockiness", "Blockiness (lower is better; HR ≈ %.2f)" % hr_block.mean())):
    lines = ["| A − B | Mean diff. | 95% CI | Wilcoxon p | A better |", "|---|---:|---:|---:|---:|"]
    lines += [paired_row(a, b, key) for a, b in PAIRS]
    display(Markdown(f"**Paired: {title}**\n\n" + "\n".join(lines)))

In [ ]:
# Distributions: PSNR vs LPIPS per method, and blockiness.
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for k, m in enumerate(METHODS):
    axes[0].scatter(values(m, "psnr"), values(m, "lpips"), s=10, alpha=0.6, color=f"C{k}", label=m)
axes[0].set(xlabel="HR PSNR (dB)", ylabel="LPIPS (lower is better)", title="Per image: PSNR vs LPIPS")
axes[0].legend(fontsize=7)
for k, m in enumerate(METHODS):
    axes[1].hist(values(m, "lpips"), bins=25, alpha=0.45, color=f"C{k}", label=m)
axes[1].set(xlabel="LPIPS", ylabel="images", title="LPIPS per method")
axes[1].legend(fontsize=7)
data = [hr_block] + [values(m, "blockiness") for m in METHODS]
axes[2].boxplot(data, tick_labels=["HR"] + [m.replace("CSGM, ", "").replace(" label", "") for m in METHODS], showfliers=False)
axes[2].axhline(1.0, color="gray", linestyle="--", linewidth=1)
axes[2].set(ylabel="blockiness (≈ 1 = no tile seams)", title="Tile-seam blockiness")
axes[2].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

# Visual examples: the first 4 images of the fixed test sample (not chosen by result).
CROP = 80


def detail_crop(hr):
    g = np.abs(np.diff(hr.mean(-1), axis=0))[:, :-1] + np.abs(np.diff(hr.mean(-1), axis=1))[:-1, :]
    pos = [(r, c) for r in range(0, 256 - CROP, 16) for c in range(0, 256 - CROP, 16)]
    return max(pos, key=lambda p: g[p[0]:p[0] + CROP, p[1]:p[1] + CROP].sum())


def bp_image(i, method):
    with np.load(POST_DIR / f"img{i:05d}.npz") as data:
        return data[method.replace(", ", "_").replace(" + ", "_").replace(" ", "_")]


show_img = lambda x: (np.clip(x, -1, 1) + 1) / 2
for i in images[:4]:
    shown = [("bicubic", bicubic_cache[i]), ("bicubic + BP", bp_image(i, "bicubic + BP")),
             ("CSGM, correct label", xhat_cache[(i, "oracle")]), ("CSGM, correct label + BP", bp_image(i, "CSGM, correct label + BP"))]
    r, c = detail_crop(hr_cache[i])
    fig, axes = plt.subplots(2, len(shown) + 1, figsize=(4 * (len(shown) + 1), 8.6))
    axes[0, 0].imshow(show_img(hr_cache[i]))
    axes[0, 0].add_patch(plt.Rectangle((c, r), CROP, CROP, fill=False, color="yellow", linewidth=1.5))
    axes[0, 0].set_title(f"Ground-truth HR, test image {i}\n{class_name(best[(i, 'oracle')]['true_label'])}", fontsize=9)
    axes[1, 0].imshow(show_img(hr_cache[i][r:r + CROP, c:c + CROP]), interpolation="nearest")
    axes[1, 0].set_title("HR (crop)", fontsize=9)
    for k, (m, x) in enumerate(shown, start=1):
        mm = post[i][m]
        axes[0, k].imshow(show_img(x))
        axes[0, k].set_title(f"{m}\nPSNR {mm['psnr']:.2f} dB | LPIPS {mm['lpips']:.3f} | block. {mm['blockiness']:.2f}", fontsize=8)
        axes[1, k].imshow(show_img(x[r:r + CROP, c:c + CROP]), interpolation="nearest")
        axes[1, k].set_title(f"{m} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.9, wspace=0.04, hspace=0.2)
    plt.show()

## Part 2 notes (100 test images; frozen CSGM outputs + parameter-free post-processing; written after the run)

**Checks passed.** Stored LR = A_sr4(HR) (max difference 1.8e-7); CSGM PSNR here equals Part 1 exactly; LR loss after every back-projection ≤ 3e-10.

**1. Back-projection helps every method on every image.**
- Bicubic + BP vs bicubic: **+0.56 dB** PSNR and **−0.054** LPIPS, better on **100/100** images for both. Plain bicubic doesn't reproduce y under our antialiased operator (LR loss 13.8), so the classical baseline to beat is **bicubic + BP: 25.57 dB, LPIPS 0.358**.
- CSGM + BP vs CSGM (correct label): **+1.05 dB** and **−0.080** LPIPS, 100/100 for both; it also lowers blockiness (1.37 → 1.26, 100/100).

**2. PSNR and LPIPS disagree: the generator prior helps perceptually but not on PSNR.**
- PSNR: CSGM + BP is **2.82 dB below** bicubic + BP (CI −3.16 to −2.48), worse on **all 100** images.
- LPIPS: CSGM + BP is **better** than bicubic + BP by **−0.074** (CI −0.098 to −0.049, p = 2.5e-7), on **76/100** images. Even without BP, CSGM beats bicubic on LPIPS (−0.048, 67/100).
- **CSGM + BP has the best mean LPIPS of all methods (0.284).** This confirms on the test set what the 4 development images suggested (notebook 04, Check 1).

**3. The generator's 16-pixel tile seams are real and measurable.**
- Blockiness: HR 1.01, bicubic 1.01, bicubic + BP 1.01; **CSGM 1.37** (correct label) and **1.46** (wrong label); BP reduces but doesn't remove them (1.26 / 1.32). CSGM + BP is blockier than bicubic + BP on **93/100** images.
- So the perceptual gain coexists with a visible artifact that full-image LPIPS doesn't penalize much. The zoomed crops (e.g. test image 5529, hatchet: the logs' bark becomes painted 16×16 tiles) show both: more plausible texture than bicubic at full size, tiles at pixel level.

**4. Label effect (correct vs wrong label), after BP.**
- PSNR: no difference (−0.06 dB, CI −0.34 to +0.22; p = 0.37), as in Part 1.
- LPIPS: correct label slightly better, **−0.018** (CI −0.032 to −0.004; p = 0.03; 54/100).
- Blockiness: correct label less blocky, **−0.059** (p = 4e-4; 65/100).
- So the class label has a small **perceptual** effect that PSNR doesn't show. Caveat: several metrics × comparisons, no multiple-comparison correction; the LPIPS effect is small and borderline.

**For the report.** Report all three metrics side by side, because they rank the methods differently:
- PSNR: bicubic + BP > bicubic > CSGM + BP > CSGM.
- LPIPS: CSGM + BP > bicubic + BP ≈ CSGM > bicubic.
- Blockiness: bicubic (+ BP) ≈ HR, CSGM worst.

**Reminder.** These are test results. They must not be used to choose or tune the IADrift variant (that's frozen on development images in `docs/PROTOCOL.md`).

---
# Part 3 (added 2026-10-05): IADrift on the test set (M5), analysed as pre-registered

> **[REPORT: core]** **Primary table** (4 pre-registered tests, Holm-corrected). **Main table:** all methods incl. IADrift @100 / @1000, raw and + BP. **Figures:** per-image IADrift vs CSGM scatter (PSNR, LPIPS), LR-loss trace, blockiness. **Figure:** visual examples with crops (first 4 test images).

**What was run** (`docs/PROTOCOL.md`, frozen 2026-10-05 before any IADrift test run; runner `scripts/run_sr4_iadrift_test.py`):
- **Start:** for each of the 100 test images and each label condition, the CSGM benchmark run with the lowest final LR loss (its ε, `noise_labels` and label).
- **IADrift:** the pretrained weights θ (fresh copy per run) and ε optimized **jointly** on ‖y − A f_θ(ε)‖²; Adam, θ lr 1e-5 (all 241 leaves), ε lr 1e-3; 1,000 steps; **outputs at the pre-declared steps 100 and 1,000** (eager generator path). One deterministic run per image and condition, 200 runs.
- **Deviation (infrastructure only):** chunk size changed from 32 to 10 runs after a segfault after 16 runs (PROTOCOL §8); method unchanged.

**What this part does** (CPU only; the GPU outputs are only loaded):
1. **Validation:** the results match the manifest plan (200 runs, test images, labels as in the CSGM benchmark), every output is finite, and the regenerated CSGM start equals the saved CSGM output (start check).
2. **Back-projection** of each IADrift output (same parameter-free operator as Part 2) and the Part 2 metrics: PSNR, LPIPS, blockiness, LR loss, band SNR.
3. **Primary analysis (PROTOCOL §6):** correct label; **IADrift @1000 + BP vs CSGM + BP** and **IADrift @100 + BP vs CSGM + BP**, each on **PSNR** and **LPIPS**: 4 tests. Paired mean difference with t-based 95% CI, Wilcoxon signed-rank p, count of images where IADrift is better, and the **Holm-adjusted** Wilcoxon p across the 4 tests.
4. **Secondary (exploratory, uncorrected):** the same without BP and with the wrong label; IADrift (+ BP) vs bicubic + BP; label effect for IADrift; blockiness; band SNR; LR loss.
5. **Visual examples:** the first 4 images of the fixed sample (not chosen by result), full image + detail crop.

**Cache:** back-projected IADrift images and metrics in `~/work/datasets/ImageNetV2/iadrift_test_postproc_v1/`.

In [ ]:
# Part 3, step 1: load and validate the frozen IADrift test run.
IAD_STEM = "phase2_sr4_iadrift_test_v1_n100_00ca59364e18"
IAD_DIR = DATA_DIR / "iadrift_test_postproc_v1"
IAD_DIR.mkdir(exist_ok=True)
iad_rows = [json.loads(line) for line in (DATA_DIR / f"{IAD_STEM}.jsonl").open() if line.strip()]
iad_plan = json.loads((DATA_DIR / f"{IAD_STEM}.manifest.json").read_text())["run_plan"]
iad = {(r["image_index"], r["condition"]): r for r in iad_rows}
assert len(iad) == len(iad_rows) == len(iad_plan) == 2 * len(images), (len(iad_rows), len(iad_plan))
for p in iad_plan:
    r = iad[(p["image_index"], p["condition"])]
    assert all(r[k] == p[k] for k in p), p
    assert r["class_label"] == best[(r["image_index"], r["condition"])]["class_label"]
    assert r["csgm_restart"] == best[(r["image_index"], r["condition"])]["restart"]
print(f"{len(iad_rows)} IADrift runs, all in the plan, starts = Part 1's best CSGM restarts")
print("all outputs finite:", all(r["step100"]["finite"] and r["step1000"]["finite"] for r in iad_rows))
print("start check, max |regenerated CSGM − saved CSGM|:", max(r["start_max_abs_diff_vs_saved_csgm"] for r in iad_rows))
print(f"time per run: median {np.median([r['elapsed_seconds'] for r in iad_rows]):.0f} s")

CKPTS = (100, 1000)
COND_NAME = {"oracle": "correct label", "wrong": "wrong label"}
IAD_METHODS = [f"IADrift @{s}, {COND_NAME[c]}{bp}" for c in ("oracle", "wrong") for s in CKPTS for bp in ("", " + BP")]

In [ ]:
# Part 3, step 2: back-project every IADrift output and compute the Part 2 metrics (cached per image).
iad_post = {}
for n, i in enumerate(images):
    path = IAD_DIR / f"img{i:05d}.npz"
    if path.exists():
        with np.load(path) as data:
            iad_post[i] = json.loads(str(data["metrics_json"]))
        continue
    y, hr = lr_all[i], hr_cache[i]
    metrics, bp_arrays = {}, {}
    for c in ("oracle", "wrong"):
        with np.load(DATA_DIR / f"{IAD_STEM}_arrays" / iad[(i, c)]["array_file"]) as data:
            for s in CKPTS:
                x = data[f"x_step{s}"]
                x_bp = backproject(x, y)
                name = f"IADrift @{s}, {COND_NAME[c]}"
                metrics[name] = evaluate(x, hr, y)
                metrics[name + " + BP"] = evaluate(x_bp, hr, y)
                bp_arrays[f"bp_{c}_step{s}"] = x_bp
    iad_post[i] = metrics
    buffer = io.BytesIO()
    np.savez_compressed(buffer, metrics_json=json.dumps(metrics), **bp_arrays)
    path.write_bytes(buffer.getvalue())
    if (n + 1) % 10 == 0:
        print(f"{n + 1}/{len(images)} images done", flush=True)

# The runner's own (eager) PSNR must match the PSNR recomputed here from the saved outputs.
d = max(abs(iad_post[i][f"IADrift @{s}, {COND_NAME[c]}"]["psnr"] - iad[(i, c)][f"step{s}"]["psnr"])
        for i in images for c in ("oracle", "wrong") for s in CKPTS)
print(f"max |PSNR here − runner PSNR|: {d:.2e} dB")
print("max LR loss after BP:", max(iad_post[i][m]["lr_loss"] for i in images for m in IAD_METHODS if m.endswith("+ BP")))
(IAD_DIR / "metrics.json").write_text(json.dumps({str(i): iad_post[i] for i in images}, indent=1))


def v(method, key, band=None):
    # Per-image values for any method of Part 2 or Part 3.
    src = iad_post if method.startswith("IADrift") else post
    return np.array([src[i][method]["band_snr_db"][band] if band else src[i][method][key] for i in images])

In [ ]:
# Part 3, step 3: the 4 pre-registered primary tests (PROTOCOL §6), correct label, Holm-corrected.
def paired_stats(a, b, key):
    d = v(a, key) - v(b, key)
    lower_better = key in ("lpips", "blockiness")
    half = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
    return {"mean": d.mean(), "lo": d.mean() - half, "hi": d.mean() + half,
            "p": stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0,
            "better": int(np.sum(d < 0 if lower_better else d > 0)), "n": len(d)}


def holm(pvalues):
    order = np.argsort(pvalues)
    adjusted, running = np.empty(len(pvalues)), 0.0
    for rank, k in enumerate(order):
        running = max(running, min(1.0, (len(pvalues) - rank) * pvalues[k]))
        adjusted[k] = running
    return adjusted


PRIMARY_TESTS = [(f"IADrift @{s}, correct label + BP", "CSGM, correct label + BP", key)
                 for s in (1000, 100) for key in ("psnr", "lpips")]
res = [paired_stats(a, b, key) for a, b, key in PRIMARY_TESTS]
adj = holm(np.array([r["p"] for r in res]))
lines = ["| # | Comparison (A − B) | Metric | Mean diff. | 95% CI | Wilcoxon p | **Holm p** | A better |",
         "|---:|---|---|---:|---:|---:|---:|---:|"]
for k, ((a, b, key), r, pa) in enumerate(zip(PRIMARY_TESTS, res, adj), start=1):
    f = "{:+.3f}" if key == "lpips" else "{:+.2f}"
    lines.append(f"| {k} | {a} − {b} | {'LPIPS (lower better)' if key == 'lpips' else 'PSNR (dB)'} | {f.format(r['mean'])} | "
                 f"[{f.format(r['lo'])}, {f.format(r['hi'])}] | {r['p']:.1e} | **{pa:.1e}** | {r['better']}/{r['n']} |")
display(Markdown("**PRIMARY ANALYSIS (pre-registered, PROTOCOL §6): 100 test images, correct label**\n\n" + "\n".join(lines)))
primary_summary = [{"comparison": f"{a} − {b}", "metric": key, **{k: float(x) for k, x in r.items()}, "holm_p": float(pa)}
                   for (a, b, key), r, pa in zip(PRIMARY_TESTS, res, adj)]
(IAD_DIR / "primary.json").write_text(json.dumps(primary_summary, indent=1))

In [ ]:
# Part 3, step 4: main table and secondary comparisons (exploratory, uncorrected).
ALL_METHODS = ["bicubic", "bicubic + BP", "CSGM, correct label", "CSGM, correct label + BP"] + \
              [m for m in IAD_METHODS if "correct" in m] + \
              ["CSGM, wrong label", "CSGM, wrong label + BP"] + [m for m in IAD_METHODS if "wrong" in m]
lines = ["| Method | Mean PSNR | PSNR (mean MSE) | LPIPS ↓ | Blockiness | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
         "|---|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
for m in ALL_METHODS:
    p = v(m, "psnr")
    lines.append(f"| {m} | {p.mean():.2f} | {psnr_of_mean_mse(p):.2f} | {v(m, 'lpips').mean():.3f} | {v(m, 'blockiness').mean():.3f} | "
                 f"{v(m, 'lr_loss').mean():.2f} | " + " | ".join(f"{v(m, None, b).mean():.2f}" for b in BAND_NAMES) + " |")
display(Markdown(f"**All methods, mean over {len(images)} test images** (HR blockiness {hr_block.mean():.3f})\n\n" + "\n".join(lines)))

SECONDARY_PAIRS = [(f"IADrift @{s}, correct label", "CSGM, correct label") for s in CKPTS] + \
                  [(f"IADrift @{s}, wrong label + BP", "CSGM, wrong label + BP") for s in CKPTS] + \
                  [(f"IADrift @{s}, correct label + BP", "bicubic + BP") for s in CKPTS] + \
                  [(f"IADrift @{s}, correct label + BP", f"IADrift @{s}, wrong label + BP") for s in CKPTS] + \
                  [("IADrift @1000, correct label + BP", "IADrift @100, correct label + BP")]
for key, title in (("psnr", "PSNR (dB)"), ("lpips", "LPIPS (lower is better)"), ("blockiness", "Blockiness (lower is better)")):
    f = "{:+.2f}" if key == "psnr" else "{:+.3f}"
    lines = ["| A − B | Mean diff. | 95% CI | Wilcoxon p | A better |", "|---|---:|---:|---:|---:|"]
    for a, b in SECONDARY_PAIRS:
        r = paired_stats(a, b, key)
        lines.append(f"| {a} − {b} | {f.format(r['mean'])} | [{f.format(r['lo'])}, {f.format(r['hi'])}] | {r['p']:.1e} | {r['better']}/{r['n']} |")
    display(Markdown(f"**Secondary (exploratory, uncorrected): {title}**\n\n" + "\n".join(lines)))

In [ ]:
# Part 3, step 5: figures.
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, key, title in ((axes[0], "psnr", "PSNR (dB)"), (axes[1], "lpips", "LPIPS")):
    base = v("CSGM, correct label + BP", key)
    for k, s in enumerate(CKPTS):
        ax.scatter(base, v(f"IADrift @{s}, correct label + BP", key), s=12, alpha=0.7, color=f"C{k}", label=f"IADrift @{s} + BP")
    lo, hi = min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set(xlabel=f"CSGM + BP: {title}", ylabel=f"IADrift + BP: {title}",
           title=f"{title} per test image (correct label)" + (": below diagonal = IADrift better" if key == "lpips" else ""))
    ax.legend(fontsize=8)
for c, color in (("oracle", "C0"), ("wrong", "C1")):
    traces = []
    for i in images:
        with np.load(DATA_DIR / f"{IAD_STEM}_arrays" / iad[(i, c)]["array_file"]) as data:
            steps, traces = data["trace_steps"], traces + [data["trace_loss"]]
    axes[2].plot(steps, np.median(traces, axis=0), color=color, label=f"median, {COND_NAME[c]}")
for s in CKPTS:
    axes[2].axvline(s, color="gray", linestyle=":", linewidth=1)
axes[2].set(yscale="log", xlabel="IADrift step", ylabel="LR loss (compiled)", title="IADrift LR loss over steps (100 test images)")
axes[2].legend(fontsize=8)
show_methods = ["bicubic + BP", "CSGM, correct label + BP", "IADrift @100, correct label + BP", "IADrift @1000, correct label + BP"]
axes[3].boxplot([hr_block] + [v(m, "blockiness") for m in show_methods], showfliers=False,
                tick_labels=["HR", "bicubic + BP", "CSGM + BP", "IADrift @100 + BP", "IADrift @1000 + BP"])
axes[3].axhline(1.0, color="gray", linestyle="--", linewidth=1)
axes[3].set(ylabel="blockiness (≈ 1 = no tile seams)", title="Tile-seam blockiness (correct label)")
axes[3].tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()

# Visual examples: the first 4 images of the fixed test sample (not chosen by result).
for i in images[:4]:
    with np.load(IAD_DIR / f"img{i:05d}.npz") as data:
        iad_bp = {s: data[f"bp_oracle_step{s}"] for s in CKPTS}
    shown = [("bicubic + BP", bp_image(i, "bicubic + BP")), ("CSGM, correct label + BP", bp_image(i, "CSGM, correct label + BP")),
             ("IADrift @100 + BP", iad_bp[100]), ("IADrift @1000 + BP", iad_bp[1000])]
    metric_of = {"bicubic + BP": post[i]["bicubic + BP"], "CSGM, correct label + BP": post[i]["CSGM, correct label + BP"],
                 "IADrift @100 + BP": iad_post[i]["IADrift @100, correct label + BP"],
                 "IADrift @1000 + BP": iad_post[i]["IADrift @1000, correct label + BP"]}
    r, c0 = detail_crop(hr_cache[i])
    fig, axes = plt.subplots(2, len(shown) + 1, figsize=(4 * (len(shown) + 1), 8.6))
    axes[0, 0].imshow(show_img(hr_cache[i]))
    axes[0, 0].add_patch(plt.Rectangle((c0, r), CROP, CROP, fill=False, color="yellow", linewidth=1.5))
    axes[0, 0].set_title(f"Ground-truth HR, test image {i}\n{class_name(best[(i, 'oracle')]['true_label'])}", fontsize=9)
    axes[1, 0].imshow(show_img(hr_cache[i][r:r + CROP, c0:c0 + CROP]), interpolation="nearest")
    axes[1, 0].set_title("HR (crop)", fontsize=9)
    for k, (m, x) in enumerate(shown, start=1):
        mm = metric_of[m]
        axes[0, k].imshow(show_img(x))
        axes[0, k].set_title(f"{m}\nPSNR {mm['psnr']:.2f} dB | LPIPS {mm['lpips']:.3f} | block. {mm['blockiness']:.2f}", fontsize=8)
        axes[1, k].imshow(show_img(x[r:r + CROP, c0:c0 + CROP]), interpolation="nearest")
        axes[1, k].set_title(f"{m} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.9, wspace=0.04, hspace=0.2)
    plt.show()

## Part 3 conclusions (IADrift on 100 test images; pre-registered analysis; written after the run)

**Checks passed.** 200/200 runs match the frozen plan; every output is finite; the regenerated CSGM start equals the saved CSGM output exactly (difference 0.0); PSNR recomputed here equals the runner's; LR loss after BP ≤ 2e-10. One infrastructure deviation (chunk size, PROTOCOL §8), method unchanged.

**Primary results (correct label; Holm-corrected over 4 tests).**

| | PSNR vs CSGM + BP | LPIPS vs CSGM + BP |
|---|---|---|
| **IADrift @1000 + BP** (as specified) | **−2.92 dB** [−3.18, −2.66], Holm p 1.6e-17, better on 0/100 | **+0.082 (worse)** [+0.066, +0.097], Holm p 1.0e-14, better on 12/100 |
| **IADrift @100 + BP** (early-stopped) | **−0.12 dB** [−0.20, −0.04], Holm p 1.4e-3, better on 33/100 | **−0.008 (better)** [−0.013, −0.004], Holm p 3.8e-5, better on 72/100 |

1. **IADrift as specified (1,000 steps) is clearly worse than CSGM on both primary metrics.** All four primary tests are significant after Holm correction.
2. **Early-stopped IADrift (100 steps) trades a very small PSNR loss (−0.12 dB) for a small perceptual gain (LPIPS −0.008).** IADrift @100 + BP has the **lowest mean LPIPS of all methods (0.276** vs CSGM + BP 0.284, bicubic + BP 0.358). The effects are statistically clear but small in size.

**Secondary findings (exploratory, uncorrected).**
- **Without BP, IADrift @100 beats CSGM on both metrics:** +0.28 dB (81/100), LPIPS −0.028 (95/100). Back-projection removes most of that advantage. Most of what 100 adaptation steps add is a better measurement fit (LR loss 61 → 33), which BP gives CSGM anyway. This replicates development Check 1 on the test set.
- **More adaptation hurts:** IADrift @1000 + BP vs @100 + BP is −2.80 dB and LPIPS +0.090 (worse on 92/100). The LR loss keeps falling (median about 1 at step 1,000), so the adaptation fits y almost exactly while HR quality falls. This is the ill-posedness pattern of Checks 2 and 6.
- **The tile seams disappear with long adaptation:** blockiness 1.26 (CSGM + BP) → 1.24 (@100) → **1.04 (@1000; HR 1.01)**. The weights learn to hide the 16-pixel grid (as in Check 6), but replace it with dense high-frequency noise: very-high-band SNR −4.1 → −8.2 dB; visible as colour speckle in the crops (e.g. test image 5529, hatchet).
- **Against bicubic + BP:** every IADrift variant is far below on PSNR (−2.94 dB at step 100). On LPIPS, IADrift @100 + BP is better (−0.082, 75/100), and @1000 + BP is no different.
- **Label:** still no PSNR effect for IADrift; small LPIPS / blockiness advantages for the correct label (e.g. @1000 + BP LPIPS −0.028, p = 0.017), the same direction as for CSGM. The wrong-label condition shows the same pattern as the correct one.

**Bottom line for the report.** On 100 held-out test images, the IAGAN-style joint adaptation of the drift generator does **not** improve on CSGM in PSNR. With early stopping, it gives a small, significant perceptual (LPIPS) gain at a negligible PSNR cost. Run as specified, it degrades both, because the SR loss leaves the high frequencies unconstrained and the adapted weights fill them with noise. This matches the development diagnosis: the drift generator *can* represent the images when every frequency is constrained (Check 6), so the failure is ill-posedness, not representation.